# Historical results and comparison

Results below are archived from the original experiments for comparison. The refactored pipeline corrected synchronization and evaluation details but has not been rerun; these are historical measurements, not reproduced benchmarks.

All tables are static snapshots of the CSVs, readable without executing any cells.

## Compared methods

- **Classical DSP:** pilot-only synchronization, APP demapping, and LDPC decoding.
- **DeepRx (CNN):** the project's historical convolutional baseline.
- **DAT (attention):** its historical attention baseline with higher reported compute than Hybrid LA.
- **Hybrid LA (ours):** explicit physical correction with lightweight learned residual processing.

The current package implements the hybrid and classical receiver. Baseline names refer to the original project's implementations.

## Fresh-seed BER comparison

1,024 frames per SNR; data seed 999, noise seed 888.

| Method | Post-FEC BER @ 6 dB | Post-FEC BER @ 12 dB | Approx. MFLOPs |
| --- | ---: | ---: | ---: |
| Classical DSP | 1.631e-01 | 8.704e-02 | — |
| DeepRx (CNN) | 1.925e-01 | 3.396e-02 | 343.72 |
| DAT (attention) | 4.243e-03 | 0 observed* | 147.81 |
| Hybrid LA (ours) | 8.393e-02 | 3.025e-02 | 72.57 |

**Key takeaway:** Hybrid LA historically achieved lower BER than the CNN baseline at both displayed fresh-seed operating points, using about **4.7× fewer reported FLOPs** than DeepRx. DAT recorded the strongest decoding results, with higher reported compute and neural-forward latency than Hybrid LA. The hybrid targets a **performance/efficiency trade-off**, rather than the lowest BER.

\* “0 observed” means zero errors in the finite evaluation sample, not zero error probability. No confidence intervals are inferred. “—” means unreported. Fresh-seed BLER was not printed.

The [benchmark CSV](../results/benchmark_results.csv) contains the full historical 0–12 dB sweep in 2 dB steps.

## Complexity comparison

| Method | Parameters | Approx. MFLOPs | ms/sample |
| --- | ---: | ---: | ---: |
| Hybrid LA (ours) | 276,888 | 72.57 | 0.08764 |
| DeepRx (CNN) | 297,924 | 343.72 | 0.12247 |
| DAT (attention) | 289,652 | 147.81 | 0.45562 |

Approximate FLOPs are the original THOP report (twice counted MACs); unsupported operations may be omitted. They are not full receiver FLOPs. Latency is CUDA neural forward time/sample at batch size 128, aggregated over the fresh-seed sweep with the first two batches discarded. It excludes generation and LDPC. GPU models were not recorded in these outputs, so do not interpret timings as a controlled hardware ranking. Classical cost was not reported.

Source: [complexity CSV](../results/complexity_results.csv).

## Original-seed results — provenance

2,000 frames per SNR; data and noise seed 46, also used in training. This is a separate protocol from the primary fresh-seed comparison.

| Method | Post-FEC BER @ 6 dB | Post-FEC BER @ 12 dB | BLER @ 12 dB |
| --- | ---: | ---: | ---: |
| Classical DSP | 1.594e-01 | 8.593e-02 | 4.020e-01 |
| DeepRx (CNN) | 1.477e-01 | 1.004e-02 | 7.400e-02 |
| DAT (attention) | 3.037e-03 | 0 observed* | 0 observed* |
| Hybrid LA (ours) | 3.765e-02 | 5.513e-03 | 1.050e-02 |

\* Zeros again denote no observed errors.

### Protocol details

- `original_seed`: 2,000 frames/SNR, data and noise seed 46, also used in training. Post-FEC metrics at 0–12 dB in 1 dB steps; pre-FEC BER only in 2 dB steps.
- `fresh_seed`: 1,024 frames/SNR, data seed 999 and noise seed 888, at 0–12 dB in 2 dB steps. Only post-FEC BER was printed.
- Historical neural decoding used `10**(Eb/N0 / 10)/4` above 6 dB and `1 + Eb/N0*0.5` otherwise. Current evaluation uses one fixed multiplier, default 1.0.
- The hybrid measurements predate the corrected CFO/time reference and independent training streams. The fresh-seed comparison is not a rerun of those corrections.
- Classical curves are taken consistently from notebook 09. DAT's separately printed classical values differ slightly; they are not averaged or substituted.
- Zero recorded errors are finite-sample observations, not zero error probability.

Excluded: smart routing/classical-estimator injection (including 2.734e−4 at 12 dB), unrelated architectures/ablations, and conflicting summary-plot claims. No values were extrapolated.

The CSVs retain printed precision, source notebook/cell (zero-based), and Git commit. Blank fields mean unreported values. The original-seed CSV includes post-FEC BER/BLER at 1 dB steps and pre-FEC BER at 2 dB steps.

## Interpretation and limitations

The historical hybrid results illustrate a useful decoding-performance/compute trade-off relative to the CNN baseline; DAT delivered stronger decoding results at greater reported compute. The original-seed/fresh-seed gap limits generalization claims, and the timing records do not establish a hardware-controlled speedup.

These observations cover synthetic AWGN, phase offset, and CFO only. Performance of the corrected pipeline remains unmeasured.